<a href="https://colab.research.google.com/github/nuvoledicaffe/Deep-Learning-Project/blob/main/MSEL_1L_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import copy

# Compare MSE L1 and MSE L2
# 1. DEVICE SETUP
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. DATALOADERS
def get_dataloaders(batch_size=128):
    cpu_transform = T.Compose([T.ToTensor()])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=cpu_transform)
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=cpu_transform)

    train_loader = torch.utils.data.DataLoader(
        trainset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True
    )
    test_loader = torch.utils.data.DataLoader(
        testset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )
    return train_loader, test_loader

# GPU Augmentation (optimized for T4/L4)
gpu_augmentations = nn.Sequential(
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
).to(device)

# 3. RESNET-18 CONFIGURATION (Optimized for CIFAR-10)
def get_model():
    model = torchvision.models.resnet18(weights=None)
    # Modification for 32x32 input: first layer 3x3 instead of 7x7 and no MaxPool
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 10)
    return model.to(device)

# 4. QUANTIZATION (INT4)
def apply_quantization(model, bits=4):
    source_model = model.module if isinstance(model, nn.DataParallel) else model
    q_model = copy.deepcopy(source_model).cpu()
    q_min, q_max = -2**(bits - 1), 2**(bits - 1) - 1
    with torch.no_grad():
        for param in q_model.parameters():
            if param.ndimension() > 1: # Applied only to weights (not bias/BN)
                max_val = param.abs().max()
                scale = max_val / q_max if max_val > 0 else 1.0
                param.copy_(torch.round(param / scale).clamp(q_min, q_max) * scale)
    return q_model.to(device)

# 5. MODIFIED TRAINING LOOP
def train_one_epoch(model, loader, optimizer, criterion, scaler, l1_lambda=0.0):
    model.train()
    for inputs, labels in loader:
        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        inputs = gpu_augmentations(inputs)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # ADDED MANUAL L1 REGULARIZATION
            if l1_lambda > 0:
                l1_loss = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

# 6. EXPERIMENT TO COMPARE L1 AND L2
def run_comparison_experiment(values, epochs=10):
    train_loader, test_loader = get_dataloaders(batch_size=256)
    scaler = torch.amp.GradScaler('cuda')
    results_l1 = []
    results_l2 = []

    for val in values:
        # --- L2 REGULARIZATION TEST (Classic Weight Decay) ---
        print(f"\n>>> Training L2 (WD)={val}")
        model = get_model()
        optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=val)
        for epoch in range(epochs):
            train_one_epoch(model, train_loader, optimizer, nn.CrossEntropyLoss(), scaler, l1_lambda=0.0)

        q_model = apply_quantization(model, bits=4)
        mse_l2 = sum(torch.mean((p1.to(device) - p2.to(device))**2).item()
                     for p1, p2 in zip(model.parameters(), q_model.parameters()) if p1.ndimension() > 1)
        results_l2.append({'val': val, 'mse': mse_l2})

        # --- L1 REGULARIZATION TEST ---
        print(f">>> Training L1={val}")
        model_l1 = get_model()
        optimizer_l1 = optim.SGD(model_l1.parameters(), lr=0.1, momentum=0.9, weight_decay=0.0) # Weight decay set to zero
        for epoch in range(epochs):
            train_one_epoch(model_l1, train_loader, optimizer_l1, nn.CrossEntropyLoss(), scaler, l1_lambda=val)

        q_model_l1 = apply_quantization(model_l1, bits=4)
        mse_l1 = sum(torch.mean((p1.to(device) - p2.to(device))**2).item()
                     for p1, p2 in zip(model_l1.parameters(), q_model_l1.parameters()) if p1.ndimension() > 1)
        results_l1.append({'val': val, 'mse': mse_l1})

    return results_l1, results_l2

def plot_comparison(res_l1, res_l2):
    vals = [r['val'] for r in res_l1]
    mse_l1 = [r['mse'] for r in res_l1]
    mse_l2 = [r['mse'] for r in res_l2]

    plt.figure(figsize=(10, 6))
    plt.plot(vals, mse_l1, 'b-o', linewidth=2, label='L1 Regularization')
    plt.plot(vals, mse_l2, 'r-d', linewidth=2, label='L2 Regularization (Weight Decay)')

    plt.xscale('log')
    plt.xlabel('Regularization Strength (Lambda / WD)')
    plt.ylabel('Total Quantization MSE')
    plt.title('Effect of L1 vs L2 Regularization on Quantization Error')
    plt.grid(True, which="both", linestyle="--", alpha=0.5)
    plt.legend()
    plt.show()

if __name__ == "__main__":
    test_values = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 5e-2]
    res_l1, res_l2 = run_comparison_experiment(test_values, epochs=10)
    plot_comparison(res_l1, res_l2)

KeyboardInterrupt: 